# BAH 2026: TIR Super-Resolution & Colorization Training

This notebook is designed to run the model training pipeline on Google Colab to leverage powerful GPUs.

## 1. Mount Google Drive
First, mount your Google Drive so Colab can access your files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Unzip the Project
Change the path below to match where you uploaded the `bah2026_colab.zip` file.

In [ ]:
import os
import zipfile

zip_path = '/content/drive/MyDrive/bah2026_colab.zip'
extract_dir = '/content/IR-colorization-BAH2026'

if os.path.exists(zip_path):
    print(f"Extracting {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print("Extraction complete!")
else:
    print(f"Warning: Could not find {zip_path}. Please make sure you uploaded the zip file.")

os.chdir(extract_dir)
print("Current Directory:", os.getcwd())

## 3. Install Requirements
Install required libraries (tifffile, diffusers, etc).

In [ ]:
!pip install tifffile pytorch-fid matplotlib diffusers transformers accelerate scikit-learn opencv-python

## 4. Train Super-Resolution Model
This will train the Attention-Gated U-Net for Super-Resolution (200m -> 100m TIR).

In [ ]:
!python train_sr.py --epochs 50 --batch_size 16

## 5. Enrich Patches (For SPADE & ControlNet)
Run this to generate semantic masks and Canny edges for your patches.

In [ ]:
!python scripts/enrich_patches.py --patches_dir output/patches

## 6. Train Colorization Model (Choose ONE)
You only need to use **one** of these approaches for your final submission. Pick either Option A or Option B.

### Option A: SPADE Colorization (GauGAN-based)
Fast and uses semantic masks to avoid "muddy" textures.

In [ ]:
# Uncomment to train SPADE
!python train_colorization.py --model spade --epochs 50 --batch_size 16

### Option B: ControlNet Diffusion (Stable Diffusion 1.5)
Extremely realistic, uses Canny edges, requires more GPU memory (T4/A100).

In [ ]:
# Uncomment to train ControlNet
# !python train_controlnet.py --model_id "runwayml/stable-diffusion-v1-5" --train_batch_size 2 --max_train_steps 5000

## 7. Run Inference
Run the end-to-end pipeline on the downscaled 200m TIR patches.

In [ ]:
# Option A (SPADE Inference)
!python inference.py --model spade --color_weights weights/best_spade_color_model.pth

# Option B (ControlNet Inference)
# !python inference_controlnet.py --controlnet_dir weights/controlnet --num_inference_steps 8

## 8. Save Weights Back to Drive
Save the generated weights folder back to your Google Drive.

In [ ]:
import os
drive_weights_path = '/content/drive/MyDrive/BAH2026_Weights'
if not os.path.exists(drive_weights_path):
    os.makedirs(drive_weights_path)
if os.path.exists('weights'):
    !cp -r weights/* {drive_weights_path}/
    print(f"Weights successfully saved to {drive_weights_path}")
else:
    print("Weights folder not found.")